# Journal de traitement — nomenclature nationale

**Pharmeazor · groupe INNOVIX · M1 IA, Université Constantine 2**

Ce carnet retrace **chaque étape** du traitement des données, de la lecture du
fichier du ministère aux livrables finaux.

Chaque anomalie y figure en trois temps :

| | |
|---|---|
| **Détection** | le code qui l'a trouvée |
| **Mesure** | combien de cas, lesquels |
| **Décision** | ce qu'on en fait, et pourquoi |

Rien n'est recopié à la main : les cellules relisent les fichiers du dépôt.
Les chiffres affichés sont donc ceux du jour où le carnet est exécuté.

> **Règle suivie partout : on ne corrige que ce qu'on peut prouver.**
> Un trou déclaré est honnête ; un trou comblé au jugé est un mensonge
> qui ne se voit pas.

In [1]:
import sys, csv, re, json
from pathlib import Path
from collections import Counter, defaultdict

RACINE = Path.cwd()
while not (RACINE / "src" / "pharmeazor").exists() and RACINE != RACINE.parent:
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE / "src"))

import pandas as pd
from pharmeazor.normalize.dosage import parse_strength, parse_packaging
from pharmeazor.normalize.substance import parse_inn
from pharmeazor.normalize.catalogue import form_family, Catalogue

REFERENCE = RACINE / "data" / "reference"
print("racine du projet :", RACINE.name)

racine du projet : pharmeazor


---

## 1. D'où viennent les données

Le ministère de l'Industrie pharmaceutique publie la **nomenclature nationale**
des produits enregistrés, un classeur Excel republié tous les deux mois.

Nous en détenons **cinq versions datées**. C'est ce qui permet de comparer une
version à la suivante et de repérer ce qui a changé.

In [2]:
classeurs = sorted((RACINE / "data" / "raw" / "nomenclature").glob("*.xlsx"))
for c in classeurs:
    print(f"  {c.name:<34} {c.stat().st_size/1048576:.1f} Mo")
print(f"\n  {len(classeurs)} versions ; la chaîne retient la plus récente :")
print(f"  {classeurs[-1].name if classeurs else '(aucune)'}")

  nomenclature_2025-12.xlsx          1.6 Mo
  nomenclature_2026-02.xlsx          1.4 Mo
  nomenclature_2026-04.xlsx          1.4 Mo
  nomenclature_2026-06.xlsx          1.4 Mo
  nomenclature_2026-08.xlsx          1.2 Mo

  5 versions ; la chaîne retient la plus récente :
  nomenclature_2026-08.xlsx


### Deux jeux de livrables, volontairement

| | Molécules | Produits |
|---|---:|---:|
| **semaine 1** | 3 | 284 |
| **élargi** | 38 | 911 |

Les deux coexistent. Le rapport de la semaine 1 annonce des chiffres précis à
l'enseignant : les écraser ferait mentir le rapport.

In [3]:
for suffixe, nom in [("_semaine1", "semaine 1"), ("_40molecules", "élargi")]:
    f = REFERENCE / f"drugs{suffixe}.csv"
    if f.exists():
        d = pd.read_csv(f, dtype=str).fillna("")
        print(f"  {nom:<12} {len(d):4d} produits, {d['inn'].nunique():3d} DCI, {len(d.columns)} colonnes")

df = pd.read_csv(REFERENCE / "drugs_40molecules.csv", dtype=str).fillna("")
print(f"\n  le reste du carnet travaille sur les {len(df)} produits")

  semaine 1     284 produits,  44 DCI, 18 colonnes
  élargi        911 produits, 131 DCI, 18 colonnes

  le reste du carnet travaille sur les 911 produits


### Une ligne, dépliée

Un CSV est un tableau enregistré en texte. Voici ce que contient une ligne.

In [4]:
SENS = {
 "drug_id": "identifiant que nous créons",
 "therapeutic_class": "famille du médicament",
 "inn": "DCI : le nom de la substance",
 "brand_name": "le nom sur la boîte",
 "registration_no": "numéro d'enregistrement officiel",
 "form": "comprimé, sirop, suppositoire…",
 "strength_raw": "le dosage, TEL QUE le ministère l'écrit",
 "packaging_raw": "le conditionnement, tel quel",
 "list_class": "classement réglementaire",
 "status": "code du ministère, non documenté",
}
for k, v in df.iloc[0].items():
    note = SENS.get(k, "")
    print(f"  {k:<20} {(v or '(vide)'):<28} {note}")

  drug_id              DZ-0001                      identifiant que nous créons
  therapeutic_class    non_classe                   famille du médicament
  inn                  ACIDE ACETYLSALICYLIQUE      DCI : le nom de la substance
  brand_name           ACEPRAL                      le nom sur la boîte
  code                 03 A 001                     
  registration_no      003/03 A 001/97/20           numéro d'enregistrement officiel
  form                 COMPRIME                     comprimé, sirop, suppositoire…
  strength_raw         500MG                        le dosage, TEL QUE le ministère l'écrit
  packaging_raw        B/20                         le conditionnement, tel quel
  list_class           (vide)                       classement réglementaire
  lab_name             GROUPE SAIDAL                
  lab_country          ALGERIE                      
  date_initial         2020-08-13                   
  date_final           2020-08-13                   
  type_cod

Les colonnes qui portent **`_raw`** gardent le texte d'origine, sans
retouche. C'est ce qui permet de toujours remonter à la source — et c'est
pourquoi certaines valeurs paraissent sales : elles le sont chez le ministère.

---

# 2. Les anomalies, une par une

## Anomalie 1 — 166 classements réglementaires vides

### Détection

In [5]:
print(df["list_class"].value_counts(dropna=False))
vides = (df["list_class"].str.strip() == "").sum()
print(f"\n  {vides} produits sur {len(df)} ({vides/len(df)*100:.1f} %) n'ont aucun classement")

list_class
LISTE I     618
            166
LISTE II    127
Name: count, dtype: int64

  166 produits sur 911 (18.2 %) n'ont aucun classement


### Mesure — qui est concerné ?

C'est la question qui tranche. Un vide n'a pas le même sens selon le produit.

In [6]:
sans = df[df["list_class"].str.strip() == ""]
print("  les molécules concernées :")
for dci, n in sans["inn"].value_counts().head(6).items():
    print(f"    {n:4d}  {dci[:56]}")

print("\n  contre-épreuve — le paracétamol associé à la codéine :")
codeine = df[df["inn"].str.contains("CODEINE", na=False)]
print(f"    {len(codeine)} produits, dont {(codeine['list_class'].str.strip() != '').sum()} CLASSÉS")

  les molécules concernées :
      88  PARACETAMOL
      28  IBUPROFENE
       8  ACIDE ACETYLSALICYLIQUE
       6  PARACETAMOL/CAFEINE
       5  PARACETAMOL/ACIDE ASCORBIQUE/MALEATE DE PHENIRAMINE
       3  PARACETAMOL/HUILE ESSENTIELLE DE PIN DE SYLVESTRE

  contre-épreuve — le paracétamol associé à la codéine :
    18 produits, dont 14 CLASSÉS


### Décision — **rien n'est rempli**

Les molécules concernées sont toutes des antalgiques **en vente libre** :
paracétamol, ibuprofène, acide acétylsalicylique.

Et la contre-épreuve est nette : le paracétamol **associé à la codéine**, lui,
est bien classé. Le vide ne veut donc pas dire « donnée manquante », il veut
dire **« hors liste »**.

Remplir ces cases inventerait un classement réglementaire que le ministère n'a
pas prononcé.

> Cette hypothèse avait été posée en semaine 1 sur 44 DCI. Elle se vérifie
> maintenant sur **131**, avec 48 cas nouveaux qui vont tous dans le même sens.

---

## Anomalie 2 — un numéro d'enregistrement pour deux produits

### Détection

In [7]:
dup = df[df["registration_no"].duplicated(keep=False) & (df["registration_no"] != "")]
print(f"  {dup['registration_no'].nunique()} numéro(s) partagé(s), {len(dup)} lignes\n")
for num in dup["registration_no"].unique():
    print(f"  {num}")
    for _, r in dup[dup["registration_no"] == num].iterrows():
        print(f"      {r['brand_name']:<26} {r['form'][:30]:<30} {r['strength_raw']}")

  1 numéro(s) partagé(s), 2 lignes

  003/13 G 181/13/24
      CLAMOXYPEN ENFANTS         POUDRE POUR SUSPENSION BUVABLE 100MG/12,5MG/ML
      CLAMOXYPEN NOURISSONS      POUDRE POUR SUSPENSION BUVABLE 100MG/12,5MG/ML


### Décision — **rien n'est supprimé**

Les deux lignes existent bien dans le fichier du ministère : ce n'est pas un
doublon de lecture. Et sur 911 produits, c'est le **seul** cas — comme en
semaine 1 sur 284.

Supprimer l'une des deux ferait disparaître un médicament réellement
enregistré. L'identifiant devient donc **composite** : numéro + nom de marque.

---

## Anomalie 3 — une date de fin antérieure au début

### Détection

In [8]:
d1 = pd.to_datetime(df["date_initial"], errors="coerce")
d2 = pd.to_datetime(df["date_final"], errors="coerce")
inc = df[(d2 < d1) & d1.notna() & d2.notna()]
print(f"  dates illisibles ou absentes : {d1.isna().sum()}")
print(f"  fin AVANT début              : {len(inc)}\n")
for _, r in inc.iterrows():
    print(f"  {r['brand_name']:<26} début {r['date_initial']}  →  fin {r['date_final']}")

print("\n  produits sans aucune date :")
for _, r in df[df["date_initial"] == ""].iterrows():
    print(f"    {r['brand_name']}")

  dates illisibles ou absentes : 5
  fin AVANT début              : 1

  APAP CAFEINE BEKER         début 2023-03-29  →  fin 2023-03-21

  produits sans aucune date :
    MEDIBREX
    FORLYSART
    BISOPROLOL HUP
    CETRIAXONE HUP
    HUCLAR


### Décision — **signalé, non corrigé**

Intervertir les deux dates serait plausible. Mais **rien n'indique laquelle est
fausse** : le début, la fin, ou les deux.

Une supposition plausible reste une supposition.

---

## Anomalie 4 — un dosage qui vaut la lettre `n`

Anomalie **nouvelle** : elle n'existait pas dans les 284 produits de la
semaine 1.

### Détection

In [9]:
illisibles = [(r["brand_name"], r["strength_raw"]) for _, r in df.iterrows()
              if r["strength_raw"] and not parse_strength(r["strength_raw"]).parsed]
print(f"  {len(illisibles)} dosage(s) illisible(s)")
for marque, valeur in illisibles:
    print(f"    {marque:<26} {valeur!r}")

  1 dosage(s) illisible(s)
    PAROL                      'n'


### Mesure — est-ce notre lecture, ou la source ?

Première hypothèse : un défaut de notre code. Vérifions dans le classeur.

In [10]:
import openpyxl
ws = openpyxl.load_workbook(classeurs[-1], read_only=True, data_only=True).worksheets[0]
for i, row in enumerate(ws.iter_rows(values_only=True), 1):
    cells = [str(c).strip() if c is not None else "" for c in row]
    if len(cells) > 6 and cells[4].upper().startswith("PAROL") and cells[3] == "PARACETAMOL":
        print(f"  ligne {i:5d}  {cells[4]:<10} dosage={cells[6]!r:<10} forme={cells[5]}")

  ligne   240  PAROL      dosage='100MG'    forme=SUPPOSITOIRE
  ligne   256  PAROL      dosage='n'        forme=SUPPOSITOIRE
  ligne   261  PAROL      dosage='200MG'    forme=SUPPOSITOIRE
  ligne   266  PAROL      dosage='300MG'    forme=SUPPOSITOIRE


### Décision — **rien n'est écrit**

La ligne 256 du classeur **officiel** contient bien la lettre `n`. Notre lecture
est fidèle ; c'est la source qui est fautive.

La valeur manquante se situe entre 100 et 200 mg — c'est **probablement**
150 mg.

> « Probablement » n'a pas sa place dans une base de sécurité médicamenteuse.
> Le cas remonte au ministère.

---

## Anomalie 5 — une même substance écrite de plusieurs façons

### Détection

In [11]:
par_cle = defaultdict(set)
for dci in df["inn"].unique():
    par_cle[parse_inn(dci).cle()].add(dci)

multi = {k: v for k, v in par_cle.items() if len(v) > 1}
print(f"  {df['inn'].nunique()} DCI distinctes dans le fichier")
print(f"  {len(multi)} substances écrites de plusieurs façons\n")
for cle, variantes in sorted(multi.items(), key=lambda x: -len(x[1]))[:3]:
    print(f"  clé « {cle} » — {len(variantes)} écritures :")
    for v in sorted(variantes)[:5]:
        print(f"      {v[:66]}")
    print()

  131 DCI distinctes dans le fichier
  20 substances écrites de plusieurs façons

  clé « GENTAMICINE » — 7 écritures :
      GENTAMICINE
      GENTAMICINE (SOUS FORME DE SULFATE)
      GENTAMICINE SULFATE
      GENTAMICINE SULFATE (GENTAMICINE)
      GENTAMICINE SULFATE EXPRIMEE EN GENTAMICINE

  clé « AMOXICILLINE » — 7 écritures :
      AMOXICILLINE SODIQUE
      AMOXICILLINE SODIQUE (AMOXICILLINE)
      AMOXICILLINE SODIQUE EXPRIME EN AMOXICILLINE
      AMOXICILLINE SOUS FORME TRIHYDRATEE
      AMOXICILLINE TRIHYDRATE EXPRIME EN AMOXICILLINE

  clé « AMOXICILLINE+ACIDE CLAVULANIQUE » — 7 écritures :
      AMOXICILLINE SODIQUE EXPRIME EN AMOXICILLINE/ACIDE CLAVULANIQUE PO
      AMOXICILLINE TRIHYDRATE EXPRIME EN AMOXICILLINE/ACIDE CLAVULANIQUE
      AMOXICILLINE TRIHYDRATEE (AMOXICILLINE)/CLAVULANATE DE POTASSIUM (
      AMOXICILLINE TRIHYDRATEE EXPRIME EN AMOXICILLINE/ACIDE CLAVULANIQU
      AMOXICILLINE TRIHYDRATEE EXPRIMEE EN AMOXICILLINE/ACIDE CLAVULANIQ



### Décision — **on regroupe par la clé, jamais par le texte**

`parse_inn` découpe une DCI en **base**, **sel** et **substances associées**.
Le regroupement se fait sur la base.

Ce qui suit montre pourquoi c'est indispensable.

In [12]:
for t in ["GENTAMICINE",
          "GENTAMICINE SULFATE",
          "GENTAMICINE (SOUS FORME DE SULFATE)",
          "AMOXICILLINE TRIHYDRATEE EXPRIME EN AMOXICILLINE",
          "AMOXICILLINE SOUS FORME TRIHYDRATEE"]:
    s = parse_inn(t)
    print(f"  {t[:50]:<52} base={s.base:<14} sel={s.sel:<12} clé={s.cle()}")

  GENTAMICINE                                          base=GENTAMICINE    sel=             clé=GENTAMICINE
  GENTAMICINE SULFATE                                  base=GENTAMICINE    sel=SULFATE      clé=GENTAMICINE
  GENTAMICINE (SOUS FORME DE SULFATE)                  base=GENTAMICINE    sel=SULFATE      clé=GENTAMICINE
  AMOXICILLINE TRIHYDRATEE EXPRIME EN AMOXICILLINE     base=AMOXICILLINE   sel=TRIHYDRATEE  clé=AMOXICILLINE
  AMOXICILLINE SOUS FORME TRIHYDRATEE                  base=AMOXICILLINE   sel=TRIHYDRATEE  clé=AMOXICILLINE


**Une association ne se confond jamais avec la substance seule.** Rapprocher
paracétamol et paracétamol/codéine ferait disparaître un opiacé.

In [13]:
for t in ["PARACETAMOL", "PARACETAMOL/CODEINE PHOSPHATE HEMIHYDRATE"]:
    s = parse_inn(t)
    print(f"  {t[:46]:<48} clé={s.cle()}")
print("\n  → deux clés différentes : ils ne seront jamais regroupés.")

  PARACETAMOL                                      clé=PARACETAMOL
  PARACETAMOL/CODEINE PHOSPHATE HEMIHYDRATE        clé=PARACETAMOL+CODEINE

  → deux clés différentes : ils ne seront jamais regroupés.


---

## Anomalie 6 — les conditionnements illisibles

C'est le plus gros morceau, et le plus instructif.

### Détection

In [14]:
lus = [parse_packaging(x) if x else None for x in df["packaging_raw"]]
non = [df["packaging_raw"].iloc[i] for i in range(len(df)) if not (lus[i] and lus[i].parsed)]
print(f"  {len(non)} conditionnements sur {len(df)} ne livrent aucune valeur")
print(f"  soit {len(non)/len(df)*100:.1f} %")
print("\n  ⚠ un seul chiffre. Il dit COMBIEN, jamais POURQUOI.")

  123 conditionnements sur 911 ne livrent aucune valeur
  soit 13.5 %

  ⚠ un seul chiffre. Il dit COMBIEN, jamais POURQUOI.


### Mesure — la méthode : remplacer les chiffres par `N`

C'est le geste qui change tout. `T/30G`, `T/50G` et `TUBE/15G` sont trois
valeurs différentes mais **une seule forme**.

In [15]:
forme = lambda s: re.sub(r"\s+", " ", re.sub(r"\d+(?:[.,]\d+)?", "N", s.upper())).strip()

familles = Counter()
for x in non:
    h = x.upper()
    if re.search(r"\bET\b|\bOU\b| - ", h):  familles["deux présentations"] += 1
    elif re.search(r"\d+\s*ML", h):          familles["un volume"] += 1
    elif re.search(r"\d+\s*G\b", h):        familles["une masse"] += 1
    else:                                    familles["autre"] += 1

for k, v in familles.most_common():
    print(f"  {v:4d}  {k}")
print("\n  → le pourcentage ne recouvrait pas une seule chose, mais plusieurs.")

    98  deux présentations
    20  un volume
     5  autre

  → le pourcentage ne recouvrait pas une seule chose, mais plusieurs.


In [16]:
print("  les formes les plus fréquentes parmi les illisibles :\n")
for f, n in Counter(forme(x) for x in non).most_common(8):
    ex = next(x for x in non if forme(x) == f)
    print(f"  {n:4d}  {f[:40]:<42} ex: {ex[:34]!r}")

  les formes les plus fréquentes parmi les illisibles :

    52  B/N ET B/N                                 ex: 'B/10 ET B/25'
     7  FL/N ML ET FL/N ML                         ex: 'FL/60 ML ET FL/30 ML'
     6  B/N AMP DE NML ET B/N AMP DE NML           ex: 'B/05 AMP DE 02ML ET B/50 AMP DE 02'
     4  B/N AMP ET B/N AMP DE NML                  ex: 'B/02 AMP ET B/50 AMP DE 3ML'
     4  B/N ET B/N ET B/N                          ex: 'B/10 ET B/15 ET B/30'
     3  FL./NML (CAISSE DE N FL.) - FL./NML (CAI   ex: 'FL./50ML (CAISSE DE 12 FL.) - FL./'
     2  B/N AMP. DE NML ET B/N AMP. DE NML         ex: 'B/05 AMP. DE 2ML ET B/50 AMP. DE 2'
     2  F L/N ML ET PIPETTE GRADUEE                ex: 'F L/40 ML ET PIPETTE GRADUEE'


### Décision 1 — **les « deux présentations » DOIVENT échouer**

Une cellule comme `B/30 ET B/90` décrit **deux tailles de boîte**. En retenir
une serait inventer.

Ce n'est pas un défaut de lecture : c'est une décision, prise en semaine 1 et
maintenue.

In [17]:
for t in ["B/30 ET B/90", "B/10 ET B/25", "B/16"]:
    p = parse_packaging(t)
    print(f"  {t:<18} lu={p.parsed!s:<6} compte={p.count}")

  B/30 ET B/90       lu=False  compte=None
  B/10 ET B/25       lu=False  compte=None
  B/16               lu=True   compte=16


### Décision 2 — les volumes et les masses étaient un **vrai** manque

Le champ `volume` existait dans le code depuis le premier jour. **Il n'était
jamais rempli.** La règle ne reconnaissait qu'une seule écriture.

Deux familles ont été mesurées avant d'écrire la moindre règle, et la plus
riche est essayée **en premier** :

In [18]:
exemples = ["B/16", "T/30G", "FL/125 ML", "B/05 AMP. DE 2ML",
            "BOITE DE 01 FLACON DE 60ML", "FL/100ML +PIPETTE GRADUEE EN KG"]
for t in exemples:
    p = parse_packaging(t)
    d = []
    if p.count is not None: d.append(f"compte={p.count}")
    if p.volume: d.append(f"mesure={p.volume}")
    print(f"  {t:<34} {', '.join(d) or '—'}")

  B/16                               compte=16
  T/30G                              mesure=30g
  FL/125 ML                          mesure=125ml
  B/05 AMP. DE 2ML                   compte=5, mesure=2ml
  BOITE DE 01 FLACON DE 60ML         compte=1, mesure=60ml
  FL/100ML +PIPETTE GRADUEE EN KG    mesure=100ml


`B/05 AMP. DE 2ML` porte **deux** informations : cinq ampoules **et** deux ml.

Si la règle du volume passait en premier, elle lirait « 2 ml » et **les cinq
ampoules disparaîtraient** — sans que personne ne s'en aperçoive, puisque le
code dirait « j'ai réussi à lire ».

### Décision 3 — l'ustensile se retire, le second composant jamais

In [19]:
for t in ["FL/100ML +PIPETTE GRADUEE EN KG",
          "B/06FL. DE PDRE + 06AMP. DE 2,5ML DE SOLV. (EPPI)"]:
    p = parse_packaging(t)
    print(f"  {t[:50]:<52} lu={p.parsed!s:<6} {p.volume or ''}")
print("\n  La pipette ne change rien au flacon : on la retire.")
print("  Le solvant est un SECOND COMPOSANT : le nécessaire échoue, et c'est voulu.")
print("  Règle : on ne coupe la queue que si elle ne porte AUCUN chiffre.")

  FL/100ML +PIPETTE GRADUEE EN KG                      lu=True   100ml
  B/06FL. DE PDRE + 06AMP. DE 2,5ML DE SOLV. (EPPI)    lu=False  

  La pipette ne change rien au flacon : on la retire.
  Le solvant est un SECOND COMPOSANT : le nécessaire échoue, et c'est voulu.
  Règle : on ne coupe la queue que si elle ne porte AUCUN chiffre.


---

# 3. La règle d'équivalence

Deux produits sont équivalents s'ils ont **la même substance**, **le même
dosage** et **la même famille de forme**.

C'est la famille de forme qui a posé le plus de problèmes.

## La voie d'administration prime sur la forme

In [20]:
for f in ["COMPRIME PELLICULE SECABLE", "SOLUTION BUVABLE",
          "SOLUTION INJECTABLE IM/IV", "COLLYRE EN SOLUTION",
          "GOUTTES OCULAIRES ET AURICULAIRES EN SOLUTION", "OVULE VAGINALE"]:
    print(f"  {f:<48} → {form_family(f)}")

  COMPRIME PELLICULE SECABLE                       → COMPRIME
  SOLUTION BUVABLE                                 → SOLUTION
  SOLUTION INJECTABLE IM/IV                        → INJECTABLE
  COLLYRE EN SOLUTION                              → COLLYRE
  GOUTTES OCULAIRES ET AURICULAIRES EN SOLUTION    → OCULAIRE
  OVULE VAGINALE                                   → VAGINAL


**Avant correction**, `SOLUTION` venait avant `INJECTABLE` dans la liste : une
solution injectable était rangée **avec les solutions buvables**.

Un groupe était déjà faux :

```
LEVOFLOXACINE 5 mg
  LEVOFLOXACINE RAZES   SOLUTION POUR PERFUSION
  LEVOQUIN              SOLUTION POUR PERFUSION
  LEVOWORLD             GOUTTES OCULAIRES ET AURICULAIRES   ← !
```

**Des gouttes pour les yeux proposées comme remplaçant d'une perfusion.**

Mesure au moment de la correction : **46 injectables** et **15 collyres**
étaient classés `SOLUTION`.

## Une faute du registre séparait le même produit

In [21]:
for f in ["POUDRE POUR SUSPENSION BUVABLE", "POUDRE POUR SUSPENSSION BUVABLE"]:
    print(f"  {f:<40} → {form_family(f)}")
print("\n  Le ministère écrit le mot des deux façons.")
print("  Sans correction, le même produit tombait dans deux familles :")
print("    BIOPAMOX  GRANULE POUR SUSPENSION BUVABLE   → SUSPENSION")
print("    HIMOXYL   POUDRE POUR SUSPENSSION BUVABLE   → POUDRE")
print("  Ils ne se rejoignaient pas.")

  POUDRE POUR SUSPENSION BUVABLE           → SUSPENSION
  POUDRE POUR SUSPENSSION BUVABLE          → SUSPENSION

  Le ministère écrit le mot des deux façons.
  Sans correction, le même produit tombait dans deux familles :
    BIOPAMOX  GRANULE POUR SUSPENSION BUVABLE   → SUSPENSION
    HIMOXYL   POUDRE POUR SUSPENSSION BUVABLE   → POUDRE
  Ils ne se rejoignaient pas.


## Les sels sont désormais visibles

Le regroupement se fait sur la **base**, pas sur le sel. Le fichier des
équivalents écrivait donc une seule DCI pour tout le groupe — ce qui était
faux.

In [22]:
eq = pd.read_csv(REFERENCE / "equivalents_40molecules.csv", dtype=str).fillna("")
g = eq[(eq["strength"] == "50mg") & (eq["form_family"] == "COMPRIME")
       & eq["inn"].str.contains("DICLOFENAC")]
if len(g):
    print(f"  DCI du groupe : {g.iloc[0]['inn']}\n")
    for _, r in g.iterrows():
        print(f"    {r['brand_name']:<22} {r['inn_produit']:<24} sel={r['sel_produit']}")
    print("\n  RAPIDUS, BIOFENAC FAST, VOTREX RAPID : le sel de potassium")
    print("  est absorbé plus vite. Les noms de marque le disent eux-mêmes.")

  DCI du groupe : DICLOFENAC SODIQUE

    BIOFENAC               DICLOFENAC SODIQUE       sel=SODIQUE
    CLOFEN                 DICLOFENAC SODIQUE       sel=SODIQUE
    DICLOFAL               DICLOFENAC SODIQUE       sel=SODIQUE
    GEOFENAC               DICLOFENAC SODIQUE       sel=SODIQUE
    XINAC-50               DICLOFENAC SODIQUE       sel=SODIQUE
    VOLTUM                 DICLOFENAC SODIQUE       sel=SODIQUE
    DICLOFAST              DICLOFENAC POTASSIUM     sel=POTASSIUM
    EFIRAP                 DICLOFENAC POTASSIUM     sel=POTASSIUM
    CATAFENAC              DICLOFENAC POTASSIUM     sel=POTASSIUM
    RAPIDUS                DICLOFENAC POTASSIQUE    sel=POTASSIQUE
    BIOFENAC FAST          DICLOFENAC POTASSIQUE    sel=POTASSIQUE
    VOTREX RAPID           DICLOFENAC POTASSIQUE    sel=POTASSIQUE

  RAPIDUS, BIOFENAC FAST, VOTREX RAPID : le sel de potassium
  est absorbé plus vite. Les noms de marque le disent eux-mêmes.


**Le regroupement n'a pas été modifié.** Décider que deux sels de diclofénac ne
sont pas interchangeables est un **jugement pharmaceutique**, pas une décision
de programme. Le pharmacien voit maintenant les deux sels et tranche lui-même.

---

# 4. État final

## Ce qui est lu

In [23]:
ds = [parse_strength(x) if x else None for x in df["strength_raw"]]
pk = [parse_packaging(x) if x else None for x in df["packaging_raw"]]
ok_d = sum(1 for x in ds if x and x.parsed)
ok_p = sum(1 for x in pk if x and x.parsed)
non = [df["packaging_raw"].iloc[i] for i in range(len(df)) if not (pk[i] and pk[i].parsed)]
voulus = [x for x in non if re.search(r"\bET\b|\bOU\b| - ", x.upper())]

print(f"  produits traités        {len(df):4d}")
print(f"  DCI distinctes          {df['inn'].nunique():4d}")
print(f"  dosages lus             {ok_d:4d}/{len(df)}  ({ok_d/len(df)*100:.1f} %)")
print(f"  conditionnements lus    {ok_p:4d}/{len(df)}  ({ok_p/len(df)*100:.1f} %)")
print(f"     échecs voulus        {len(voulus):4d}")
print(f"     formes rares         {len(non)-len(voulus):4d}")

  produits traités         911
  DCI distinctes           131
  dosages lus              910/911  (99.9 %)
  conditionnements lus     788/911  (86.5 %)
     échecs voulus          98
     formes rares           25


## Les livrables

In [24]:
for f in sorted(REFERENCE.glob("*.csv")):
    with open(f, encoding="utf-8", newline="") as fh:
        n = max(0, sum(1 for _ in csv.reader(fh)) - 1)
    print(f"  {f.name:<36} {n:5d} lignes")

  conflits_40molecules.csv                10 lignes
  conflits_semaine1.csv                    0 lignes
  desaccords_versions.csv                 13 lignes
  documents_a_collecter.csv               18 lignes
  drugs.sample.csv                         7 lignes
  drugs_40molecules.csv                  911 lignes
  drugs_semaine1.csv                     284 lignes
  equivalents_40molecules.csv            782 lignes
  equivalents_semaine1.csv               242 lignes
  interactions_40molecules.csv           256 lignes
  interactions_semaine1.csv               87 lignes
  name_index.csv                        1136 lignes
  name_index_40molecules.csv            3644 lignes


## Ce qui reste ouvert, et pourquoi

| | |
|---|---|
| Le code `F` / `I` du champ statut | aucune légende nulle part — **à demander à l'enseignant** |
| 5 produits sans aucune date | signalés, non complétés |
| Le `PAROL` dont le dosage vaut `n` | erreur du ministère, non devinée |
| ~24 conditionnements de forme rare | dont 2 où le nombre est écrit en toutes lettres |

---

## Ce que ce carnet démontre

Sur six anomalies examinées, **quatre se terminent par « rien n'est écrit »**.

C'est le cœur du travail : documenter un trou plutôt que le combler.

C'est aussi exactement ce que le système devra faire face à une question dont
la réponse n'est pas dans le corpus — dire qu'il ne sait pas, au lieu
d'inventer.